# Minimal QLoRA (Adapter-Only)\n
\n
This notebook is a placeholder skeleton for the isolated bonus experiment.\n
Production model and RAG runtime must remain unchanged.

## Execution phases\n1. Install dependencies\n2. Set reproducibility seed\n3. Load train/eval JSONL datasets\n4. Format train samples using Qwen chat template\n5. Load base model in 4-bit\n6. Apply LoRA and run short SFT training\n7. Evaluate three modes (base, base+system prompt, base+adapter)\n8. Export adapter and eval logs

In [ ]:
# 1) Install dependencies (Colab)
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
# 2) Imports and deterministic seed
import json
import random
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SYSTEM_PROMPT = "You are an AI assistant for Acibadem University. Answer briefly, stay factual, and avoid unsupported claims."

In [ ]:
# 3) Load datasets
# In Colab, upload train_30.jsonl and eval_15.jsonl to /content, or mount Drive and update paths.
TRAIN_PATH = "/content/train_30.jsonl"
EVAL_PATH = "/content/eval_15.jsonl"

train_ds = load_dataset("json", data_files=TRAIN_PATH)["train"]
eval_ds = load_dataset("json", data_files=EVAL_PATH)["train"]

print("train rows:", len(train_ds))
print("eval rows:", len(eval_ds))

In [ ]:
# 4) Qwen chat-formatting for SFT (critical)
def format_sample(sample):
    return (
        "<|im_start|>system\n"
        "You are an AI assistant for Acibadem University.\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{sample['instruction']}\n"
        f"Question: {sample['input']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{sample['output']}\n"
        "<|im_end|>"
    )

train_sft = train_ds.map(lambda x: {"text": format_sample(x)})
print(train_sft[0]["text"][:500])

In [ ]:
# 5) Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
print("Model loaded:", BASE_MODEL)

In [ ]:
# 6) LoRA config (Qwen-friendly target modules)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# Smoke-friendly config for limited Colab runtime
training_args = TrainingArguments(
    output_dir="/content/qlora_adapter_out",
    num_train_epochs=1,
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=15,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_sft,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

print("Trainer ready")

In [ ]:
# 7) Train (smoke run)
train_result = trainer.train()
print(train_result)

ADAPTER_OUT = "/content/qlora_adapter_out"
trainer.model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print("Adapter saved to:", ADAPTER_OUT)

# Optional: zip adapter for easy download
!cd /content && zip -r qlora_adapter_out.zip qlora_adapter_out > /dev/null
print("Adapter zip: /content/qlora_adapter_out.zip")

In [ ]:
# 8) Eval helpers (three modes)
def build_prompt(question, use_system_prompt=False):
    if use_system_prompt:
        return (
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"{question}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
    return (
        "<|im_start|>user\n"
        f"{question}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

@torch.no_grad()
def generate_answer(model, prompt, max_new_tokens=180):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()

In [ ]:
# 9) Load adapter on top of base model for comparison
adapter_model = PeftModel.from_pretrained(base_model, ADAPTER_OUT)
adapter_model.eval()
base_model.eval()

print("Base and adapter models ready for eval")

In [ ]:
# 10) Run evaluation and export structured logs
results = []
for row in eval_ds:
    qid = int(row["qid"])
    question = row["question"]

    base_raw = generate_answer(base_model, build_prompt(question, use_system_prompt=False))
    base_prompted = generate_answer(base_model, build_prompt(question, use_system_prompt=True))
    adapter_ans = generate_answer(adapter_model, build_prompt(question, use_system_prompt=True))

    results.append(
        {
            "qid": qid,
            "question": question,
            "expected_focus": row.get("expected_focus", ""),
            "scoring_hint": row.get("scoring_hint", ""),
            "base_answer": base_raw,
            "base_prompted_answer": base_prompted,
            "adapter_answer": adapter_ans,
            "format_ok": "TBD",
            "hallucination_flag": "TBD",
            "notes": "",
        }
    )

out_dir = Path('/content/results')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'eval_outputs_3mode.jsonl'

with out_path.open('w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Saved {len(results)} rows to {out_path}")
print(results[0])

## Copy-back checklist (manual)
After Colab run completes:

1. Download `/content/qlora_adapter_out.zip` (or `/content/qlora_adapter_out`).
2. Download `/content/results/eval_outputs_3mode.jsonl`.
3. Copy them into repo artifacts:
   - `experiments/qlora/results/artifacts/qlora_adapter_out.zip`
   - `experiments/qlora/results/artifacts/eval_outputs_3mode.jsonl`
4. Fill `experiments/qlora/results/comparison_table.md` from the JSONL outputs.
5. Update `docs/AI_TUNING_BONUS.md` with final H1/H2/H3 conclusions.